# Klasifikasi Uang Koin Rupiah - GoogLeNet (Inception) Architecture (5 Kelas)

**Cakupan Asistensi & Implementasi GoogLeNet:**
- **Arsitektur GoogLeNet (Inception):** Menerapkan arsitektur konvolusi multi-skala *Inception* (Szegedy et al., 2014) untuk klasifikasi 5 kelas uang koin Rupiah (`100`, `200`, `500`, `1000`, dan `sisi_garuda`).
- **Rasional Ilmiah Pemilihan GoogLeNet untuk Koin:**
  1. *Multi-Scale Feature Processing:* Koin memiliki fitur visual pada skala spasial yang beragam. Filter $1\times1$ dan $3\times3$ menangkap *micro-relief* angka tipis dan teks, sedangkan filter $5\times5$ dan *pooling* menangkap siluet lingkaran koin dan Lambang Garuda. Modul Inception memproses seluruh skala ini secara paralel pada lapisan yang sama.
  2. *Efisiensi Reduksi Dimensi ($1\times1$ Bottleneck):* Menggunakan konvolusi $1\times1$ sebelum filter $3\times3$ dan $5\times5$ untuk menekan beban komputasi dan parameter.
  3. *Global Average Pooling (GAP):* Menggantikan lapisan *fully-connected* masif di akhir jaringan untuk mencegah overfitting pada dataset koin (~713 citra).
- **Pipeline Data & Anti-Kebocoran:** Audit kualitas citra, deduplikasi SHA-256 + dHash 64-bit, split stratified group-aware 70/15/15, serta augmentasi rotasi 360° fisik koin.
- **Ekspor Deployment:** Konversi model terpilih ke format Keras (`.keras`) dan TensorFlow Lite (`.tflite` FP16) untuk siap diintegrasikan ke sistem serving/web app.

## 1. Setup dan Konfigurasi

Seluruh parameter penting dirangkum dalam `Config` dataclass agar terstruktur dan mudah dikonfigurasi ulang.

In [ ]:
import os, sys, io, json, math, random, shutil, hashlib, platform, subprocess, time, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["MPLCONFIGDIR"] = os.path.join(os.environ.get("TEMP", "C:/tmp"), "mpl_cache")
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x

IN_COLAB = Path('/content').is_dir()
if IN_COLAB:
    BASE_DIR = Path('/content')
else:
    ws_candidate = Path(r"C:/Users/Bunga Alyssa/OneDrive/文档/DS/smt  7/deep l/uas/frontend")
    BASE_DIR = ws_candidate if ws_candidate.exists() else Path('.').resolve()


@dataclass(frozen=True)
class Config:
    # --- data
    seed: int = 42
    classes: tuple = ('100', '200', '500', '1000', 'sisi_garuda')
    img_size: tuple = (224, 224)
    split_ratios: tuple = (0.70, 0.15, 0.15)   # train, val, test
    n_group_folds: int = 20                    # 20 fold -> 14 train, 3 val, 3 test
    near_dup_threshold: int = 6                # jarak Hamming dHash (0-64)
    resize_mode: str = 'pad'                   # 'pad' = pertahankan rasio lingkaran koin
    n_workers: int = min(8, os.cpu_count() or 2)
    # --- training
    batch_size: int = 32
    baseline_epochs: int = 15
    tuning_epochs: int = 30
    baseline_lr: float = 1e-3
    tuning_lr: float = 5e-4
    patience: int = 8
    l2_reg: float = 1e-4
    # --- lokasi dataset
    data_dir_name: str = 'dataset_koin_baru'
    data_root_override: str = os.environ.get('COIN_DATA_ROOT', '')
    quick_test: bool = os.environ.get('COIN_QUICK', '') == '1'


cfg = Config()
CLASSES = list(cfg.classes)
IMG_SIZE = tuple(cfg.img_size)
SEED = cfg.seed
random.seed(SEED)
np.random.seed(SEED)

OUT_ROOT = BASE_DIR / 'coin_processed'
REPORT_DIR = OUT_ROOT / 'eda_reports'
RESULTS = OUT_ROOT / 'training_results_googlenet'
for d in (OUT_ROOT, REPORT_DIR, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False})


def pretty(c):
    if c == 'sisi_garuda':
        return 'Sisi Garuda / Pahlawan'
    return 'Rp ' + f"{int(c):,}".replace(',', '.')


print('Konfigurasi Model GoogLeNet Koin:')
for k, v in asdict(cfg).items():
    print(f'  {k:20s} {v}')
if cfg.quick_test:
    print('\n>>> MODE UJI CEPAT AKTIF: epoch dipangkas.')

## 2. Penemuan Dataset Koin

Dataset koin dimuat dari folder `dataset_koin_baru` (5 kelas target).

In [ ]:
if cfg.data_root_override:
    DATA_ROOT = Path(cfg.data_root_override)
else:
    candidates = [
        BASE_DIR / cfg.data_dir_name,
        BASE_DIR.parent / cfg.data_dir_name,
        Path(cfg.data_dir_name),
        Path('/content') / cfg.data_dir_name
    ]
    DATA_ROOT = next((d for d in candidates if d.exists() and all((d / c).is_dir() for c in CLASSES)), None)

if DATA_ROOT is None:
    raise FileNotFoundError(f'Folder 5 kelas koin tidak ditemukan di calon lokasi: {[str(c) for c in candidates]}')

print(f'DATA_ROOT terdeteksi : {DATA_ROOT.resolve()}')
print('Subfolder kelas terverifikasi:')
for c in CLASSES:
    n_files = len(list((DATA_ROOT / c).glob('*.*')))
    print(f'  - {c:12s} ({pretty(c):<25s}): {n_files} berkas')

## 3. Inventaris Dataset dan Audit Kualitas Citra (EDA)

Pemeriksaan integritas setiap berkas citra koin:
- **Validitas & Format:** Deteksi citra rusak.
- **Blur Laplacian:** Mengukur tingkat keburaman citra koin.
- **Brightness & Contrast:** Rata-rata dan standar deviasi keabuan.
- **Glare Ratio:** Proporsi piksel jenuh ($\ge 250$) untuk mendeteksi kilau reflektif permukaan logam koin.
- **dHash (64-bit):** Sidik jari visual untuk deteksi duplikat.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def laplacian_variance(gray):
    g = gray.astype(np.float32)
    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return float(lap.var())


def dhash64(gray_img):
    a = np.asarray(gray_img.resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    h = 0
    for b in bits:
        h = (h << 1) | int(b)
    return h


def analyze_image(item):
    path, cls = item
    rec = {'path': str(path), 'filename': path.name, 'label': cls, 'valid': False, 'error': None}
    try:
        data = path.read_bytes()
        sha = hashlib.sha256(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im:
            fmt, mode0 = im.format, im.mode
            orientation = im.getexif().get(0x0112, 1)
            w, h = im.size
            if orientation in (5, 6, 7, 8):
                w, h = h, w
            if w < 1 or h < 1:
                raise ValueError('Dimensi tidak valid')
            im.draft('RGB', (448, 448))
            rgb = ImageOps.exif_transpose(im).convert('RGB')
        gray = rgb.convert('L').resize(IMG_SIZE, Image.Resampling.BOX)
        g = np.asarray(gray, dtype=np.float32)
        rec.update(valid=True, width=w, height=h, mode=mode0, format=fmt,
                   aspect_ratio=w / h, file_kb=len(data) / 1024, sha256=sha,
                   brightness=float(g.mean()), contrast=float(g.std()),
                   blur_laplacian=laplacian_variance(g),
                   glare_ratio=float((g >= 250).mean()),
                   dhash=f'{dhash64(gray):016x}')
    except Exception as e:
        rec['error'] = f'{type(e).__name__}: {e}'
    return rec


files = [(p, cls) for cls in CLASSES
         for p in sorted((DATA_ROOT / cls).rglob('*'))
         if p.is_file() and p.suffix.lower() in IMG_EXT]
if not files:
    raise RuntimeError('Tidak ada gambar koin ditemukan.')

t0 = time.time()
with ThreadPoolExecutor(cfg.n_workers) as ex:
    records = list(tqdm(ex.map(analyze_image, files), total=len(files), desc='Menganalisis citra koin'))
inventory = pd.DataFrame(records)
valid = inventory[inventory.valid].copy().reset_index(drop=True)
valid[['width', 'height']] = valid[['width', 'height']].astype(int)
valid['orientation'] = np.where(valid.width >= valid.height, 'landscape', 'portrait')
print(f'Selesai analisis dalam {time.time() - t0:.1f} detik')

n_bad = int((~inventory.valid).sum())
per_class = valid.label.value_counts().reindex(CLASSES, fill_value=0)
print(f'Total citra koin ditemukan   : {len(inventory)}')
print(f'Valid / tidak valid          : {len(valid)} / {n_bad}')
print(f'Rasio kelas terbesar/terkecil: {per_class.max() / per_class.min():.2f}')
display(per_class.rename('jumlah').to_frame().assign(persen=lambda d: (d.jumlah / d.jumlah.sum() * 100).round(1)))

In [ ]:
def load_thumb(path, max_side=480):
    with Image.open(path) as im:
        im.draft('RGB', (max_side, max_side))
        im = ImageOps.exif_transpose(im).convert('RGB')
        im.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
        return im

q = valid.copy()
q['log_blur'] = np.log10(q.blur_laplacian + 1e-6)

def iqr_fence(s, k=1.5):
    q1, q3 = s.quantile([.25, .75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

blur_lo, _ = iqr_fence(q.log_blur)
b_lo, b_hi = iqr_fence(q.brightness)
c_lo, _ = iqr_fence(q.contrast)
_, g_hi = iqr_fence(q.glare_ratio)

q['flag_blur'] = q.log_blur < blur_lo
q['flag_dark'] = q.brightness < b_lo
q['flag_bright'] = q.brightness > b_hi
q['flag_low_contrast'] = q.contrast < c_lo
q['flag_glare'] = q.glare_ratio > max(g_hi, 0.05)
q['n_flags'] = q[['flag_blur', 'flag_dark', 'flag_bright', 'flag_low_contrast', 'flag_glare']].sum(axis=1)

print(f'Total gambar dengan >=1 tanda outlier kualitas: {(q.n_flags > 0).sum()} dari {len(q)}')

## 4. Deteksi Duplikat dan Pencegahan Kebocoran Data (Data Leakage)

Pengelompokan near-duplicate menggunakan jarak Hamming dHash 64-bit ($\le 6$) agar frame dari rekaman video koin yang sama tidak bocor antar partisi train, validation, dan test.

In [ ]:
hash_nlabels = valid.groupby('sha256').label.nunique()
conflict_hashes = set(hash_nlabels[hash_nlabels > 1].index)
is_conflict = valid.sha256.isin(conflict_hashes)
is_exact_copy = valid.duplicated('sha256', keep='first') & ~is_conflict

excluded = pd.concat([
    valid[is_conflict].assign(reason='label_konflik'),
    valid[is_exact_copy].assign(reason='duplikat_persis'),
])[['path', 'label', 'sha256', 'reason']]
clean = valid[~is_conflict & ~is_exact_copy].reset_index(drop=True)
print(f'Dikeluarkan: {len(excluded)} | Citra koin unik tersisa: {len(clean)}')

POP8 = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)

def hamming_matrix(h):
    x = np.bitwise_xor(h[:, None], h[None, :])
    return POP8[x.view(np.uint8).reshape(len(h), len(h), 8)].sum(axis=2, dtype=np.int16)

h = np.array([int(x, 16) for x in clean.dhash], dtype=np.uint64)
D = hamming_matrix(h)
lab = clean.label.to_numpy(dtype=object)
same_label = (lab[:, None] == lab[None, :])
np.fill_diagonal(same_label, False)

D_same = np.where(same_label, D, 65)

from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

adj = (D_same <= cfg.near_dup_threshold)
n_groups, group_ids = connected_components(csr_matrix(adj), directed=False)
clean['group_id'] = group_ids
print(f'Total citra koin bersih       : {len(clean)}')
print(f'Total kelompok unik (anti-leak): {n_groups}')

## 5. Split Stratified dan Group-Aware (70 / 15 / 15)

`StratifiedGroupKFold` 20-fold (14 train, 3 val, 3 test) berbasis `group_id` dan `label`.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def grouped_stratified_split(frame, ratios, n_folds, seed):
    fold_counts = [r * n_folds for r in ratios]
    if any(abs(c - round(c)) > 1e-6 for c in fold_counts):
        raise ValueError(f'Rasio {ratios} tidak kompatibel dengan n_group_folds={n_folds}.')
    n_tr, n_va, _ = (round(c) for c in fold_counts)
    min_cls = frame.label.value_counts().min()
    if min_cls < n_folds:
        raise ValueError(f'Kelas terkecil hanya {min_cls} gambar unik (< {n_folds} fold).')
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold = np.empty(len(frame), dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(frame, frame.label, frame.group_id)):
        fold[idx] = k
    return np.where(fold < n_tr, 'train', np.where(fold < n_tr + n_va, 'val', 'test'))

split_df = clean.copy()
split_df['split'] = grouped_stratified_split(split_df, cfg.split_ratios, cfg.n_group_folds, SEED)

assert (split_df.groupby('group_id').split.nunique() == 1).all(), 'Ada group koin yang terpecah antar split!'
sets = {s: set(split_df.loc[split_df.split == s, 'sha256']) for s in ('train', 'val', 'test')}
assert not (sets['train'] & sets['val'] or sets['train'] & sets['test'] or sets['val'] & sets['test'])
print('OK: Verifikasi Anti-Kebocoran Lolos (Zero Leakage Antar Split)')

counts = pd.crosstab(split_df.label, split_df.split).reindex(CLASSES)[['train', 'val', 'test']]
counts['total'] = counts.sum(axis=1)
display(counts)

## 6. Preprocessing Citra Koin dan Standarisasi 224×224

Mode `pad` mempertahankan bentuk koin lingkaran asli tanpa distorsi elips.

In [ ]:
def preprocess_image(src, dst, size=IMG_SIZE, mode=cfg.resize_mode):
    with Image.open(src) as im:
        im.draft('RGB', (size[0] * 2, size[1] * 2))
        img = ImageOps.exif_transpose(im).convert('RGB')
    if mode == 'pad':
        resized = ImageOps.contain(img, size, method=Image.Resampling.LANCZOS)
        out = Image.new('RGB', size, (0, 0, 0))
        out.paste(resized, ((size[0] - resized.width) // 2, (size[1] - resized.height) // 2))
    elif mode == 'stretch':
        out = img.resize(size, Image.Resampling.LANCZOS)
    else:
        raise ValueError(f'resize_mode tidak dikenal: {mode}')
    dst.parent.mkdir(parents=True, exist_ok=True)
    out.save(dst, format='JPEG', quality=95)

for s in ('train', 'val', 'test'):
    if (OUT_ROOT / s).exists():
        shutil.rmtree(OUT_ROOT / s)

split_df = split_df.reset_index(drop=True)
split_df['processed_path'] = [
    str(OUT_ROOT / r.split / r.label / f'{i:05d}_{r.sha256[:10]}.jpg')
    for i, r in enumerate(split_df.itertuples())
]

jobs = list(zip(split_df.path, map(Path, split_df.processed_path)))
t0 = time.time()
with ThreadPoolExecutor(cfg.n_workers) as ex:
    list(tqdm(ex.map(lambda a: preprocess_image(*a), jobs), total=len(jobs), desc='Preprocessing koin'))
print(f'Selesai preprocessing dalam {time.time() - t0:.1f} detik')

for s in ('train', 'val', 'test'):
    n_disk = sum(1 for _ in (OUT_ROOT / s).rglob('*.jpg'))
    assert n_disk == (split_df.split == s).sum(), f'Jumlah berkas pada disk {s} tidak cocok'
print('Validasi preprocessing SUKSES: seluruh citra 224x224 RGB.')

## 7. Pipeline `tf.data` dan Augmentasi Citra Koin

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers

tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
print('TensorFlow Version :', tf.__version__)
print('Perangkat GPU      :', [g.name for g in tf.config.list_physical_devices('GPU')] or 'Tidak ada GPU (CPU aktif)')


def make_ds(split, training):
    raw = tf.keras.utils.image_dataset_from_directory(
        str(OUT_ROOT / split), labels='inferred', label_mode='int', class_names=CLASSES,
        image_size=IMG_SIZE, batch_size=None, shuffle=False
    )
    paths = list(raw.file_paths)
    ds = raw.cache()
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(cfg.batch_size).prefetch(AUTOTUNE), paths


train_ds, train_paths = make_ds('train', True)
val_ds, val_paths = make_ds('val', False)
test_ds, test_paths = make_ds('test', False)

train_counts = split_df[split_df.split == 'train'].label.value_counts().reindex(CLASSES)
imbalance = train_counts.max() / train_counts.min()
USE_CLASS_WEIGHT = bool(imbalance > 1.5)
CLASS_WEIGHT = {i: float(train_counts.sum() / (len(CLASSES) * train_counts[c])) for i, c in enumerate(CLASSES)}
print(f'Rasio ketidakseimbangan train = {imbalance:.2f} -> Class weight {"DIPAKAI" if USE_CLASS_WEIGHT else "TIDAK DIPAKAI"}')


def build_augmentation():
    return tf.keras.Sequential([
        layers.RandomRotation(0.50, fill_mode='constant', fill_value=0.0),       # Rotasi penuh 360 derajat
        layers.RandomZoom(0.10, fill_mode='constant', fill_value=0.0),
        layers.RandomTranslation(0.05, 0.05, fill_mode='constant', fill_value=0.0),
        layers.RandomContrast(0.20),
        layers.RandomBrightness(0.20, value_range=(0, 255)),
    ], name='augmentation_googlenet_coin')

## 8. Arsitektur GoogLeNet (Inception Architecture)

Arsitektur GoogLeNet / Inception memiliki keunggulan utama dalam **pemrosesan multi-skala**:
1. **Stem Network:**
   - `Conv2D(64, 7x7, strides=2)` + `MaxPool(3x3, strides=2)`
   - `Conv2D(64, 1x1)` + `Conv2D(192, 3x3)` + `MaxPool(3x3, strides=2)`
2. **Inception Modules:**
   Setiap modul Inception terdiri dari 4 cabang paralel:
   - **Branch 1:** $1 \times 1$ Konvolusi (ekstraksi fitur lokal berdimensi rendah).
   - **Branch 2:** $1 \times 1$ Reduksi $\to$ $3 \times 3$ Konvolusi (ekstraksi tekstur dan relief nominal).
   - **Branch 3:** $1 \times 1$ Reduksi $\to$ $5 \times 5$ Konvolusi (diimplementasikan via 2x $3\times3$ bertingkat untuk efisiensi parameter).
   - **Branch 4:** $3 \times 3$ Max Pooling $\to$ $1 \times 1$ Proyeksi konvolusi.
   - Hasil dari keempat branch digabungkan (*concatenate*) sepanjang channel.
3. **Tahapan Inception:**
   - Stage 3: Inception 3a, Inception 3b $\to$ MaxPool
   - Stage 4: Inception 4a, Inception 4b $\to$ MaxPool
   - Stage 5: Inception 5a, Inception 5b
4. **Classifier Head:**
   - `GlobalAveragePooling2D()`
   - `Dropout(0.4)`
   - `Dense(128, activation='relu')`
   - `Dense(5, activation='softmax')`

In [ ]:
def inception_module(x, f_1x1, f_3x3_r, f_3x3, f_5x5_r, f_5x5, f_pool_proj, l2=1e-4, name='inception'):
    """Modul Inception GoogLeNet dengan 4 branch konvolusi paralel."""
    # Branch 1: 1x1 Conv
    b1 = layers.Conv2D(f_1x1, (1, 1), padding='same', activation='relu',
                       kernel_regularizer=regularizers.l2(l2), name=f'{name}_1x1')(x)
    b1 = layers.BatchNormalization(name=f'{name}_bn1')(b1)

    # Branch 2: 1x1 Conv -> 3x3 Conv
    b2 = layers.Conv2D(f_3x3_r, (1, 1), padding='same', activation='relu',
                       kernel_regularizer=regularizers.l2(l2), name=f'{name}_3x3_r')(x)
    b2 = layers.BatchNormalization(name=f'{name}_bn2_1')(b2)
    b2 = layers.Conv2D(f_3x3, (3, 3), padding='same', activation='relu',
                       kernel_regularizer=regularizers.l2(l2), name=f'{name}_3x3')(b2)
    b2 = layers.BatchNormalization(name=f'{name}_bn2_2')(b2)

    # Branch 3: 1x1 Conv -> 5x5 Conv (2x 3x3 setara 5x5 Inception-v2/v3 efisien)
    b3 = layers.Conv2D(f_5x5_r, (1, 1), padding='same', activation='relu',
                       kernel_regularizer=regularizers.l2(l2), name=f'{name}_5x5_r')(x)
    b3 = layers.BatchNormalization(name=f'{name}_bn3_1')(b3)
    b3 = layers.Conv2D(f_5x5, (3, 3), padding='same', activation='relu',
                       kernel_regularizer=regularizers.l2(l2), name=f'{name}_5x5_a')(b3)
    b3 = layers.BatchNormalization(name=f'{name}_bn3_2')(b3)
    b3 = layers.Conv2D(f_5x5, (3, 3), padding='same', activation='relu',
                       kernel_regularizer=regularizers.l2(l2), name=f'{name}_5x5_b')(b3)
    b3 = layers.BatchNormalization(name=f'{name}_bn3_3')(b3)

    # Branch 4: 3x3 MaxPool -> 1x1 Conv
    b4 = layers.MaxPooling2D((3, 3), strides=(1, 1), padding='same', name=f'{name}_pool')(x)
    b4 = layers.Conv2D(f_pool_proj, (1, 1), padding='same', activation='relu',
                       kernel_regularizer=regularizers.l2(l2), name=f'{name}_pool_proj')(b4)
    b4 = layers.BatchNormalization(name=f'{name}_bn4')(b4)

    return layers.concatenate([b1, b2, b3, b4], axis=-1, name=f'{name}_concat')


def build_googlenet_coin(use_aug=False, l2=cfg.l2_reg):
    """Membangun arsitektur GoogLeNet lengkap yang disesuaikan untuk klasifikasi koin."""
    inp = layers.Input(shape=(*IMG_SIZE, 3), name='coin_image')
    x = build_augmentation()(inp) if use_aug else inp
    x = layers.Rescaling(1. / 255, name='rescale_0_1')(x)

    # Stem Network
    x = layers.Conv2D(64, (7, 7), strides=(2, 2), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(l2), name='stem_conv1')(x)
    x = layers.BatchNormalization(name='stem_bn1')(x)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same', name='stem_pool1')(x)

    x = layers.Conv2D(64, (1, 1), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(l2), name='stem_conv2')(x)
    x = layers.Conv2D(192, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(l2), name='stem_conv3')(x)
    x = layers.BatchNormalization(name='stem_bn2')(x)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same', name='stem_pool2')(x)

    # Inception Stage 3
    x = inception_module(x, 64, 96, 128, 16, 32, 32, l2=l2, name='inc_3a')
    x = inception_module(x, 128, 128, 192, 32, 96, 64, l2=l2, name='inc_3b')
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same', name='pool3')(x)

    # Inception Stage 4
    x = inception_module(x, 192, 96, 208, 16, 48, 64, l2=l2, name='inc_4a')
    x = inception_module(x, 160, 112, 224, 24, 64, 64, l2=l2, name='inc_4b')
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same', name='pool4')(x)

    # Inception Stage 5
    x = inception_module(x, 256, 160, 320, 32, 128, 128, l2=l2, name='inc_5a')
    x = inception_module(x, 384, 192, 384, 48, 128, 128, l2=l2, name='inc_5b')

    # Classifier Head
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.4, name='dropout')(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2), name='dense_128')(x)
    out = layers.Dense(len(CLASSES), activation='softmax', name='kelas')(x)

    return tf.keras.Model(inp, out, name='googlenet_coin')


def compile_model(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(2, name='top2')]
    )
    return model


demo_model = build_googlenet_coin()
print('GoogLeNet Koin Berhasil Dibangun! Total Parameter:', f'{demo_model.count_params():,}')
demo_model.summary(line_length=110)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report


def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return centre - half, centre + half


def plot_history(history_df, title, filename):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip(axes, ['accuracy', 'loss']):
        ax.plot(history_df.index + 1, history_df[m], label='train', color='#4C78A8', lw=2)
        ax.plot(history_df.index + 1, history_df['val_' + m], label='validation', color='#E45756', lw=2)
        ax.set(xlabel='Epoch', ylabel=m, title=f'{title} - {m.capitalize()}')
        ax.legend()
    plt.tight_layout()
    fig.savefig(RESULTS / filename, dpi=160)
    plt.show()


def train_experiment(name, title, use_aug, epochs, lr, early_stop):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = compile_model(build_googlenet_coin(use_aug=use_aug), lr)
    cbs = [tf.keras.callbacks.TerminateOnNaN()]
    if early_stop:
        cbs += [
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg.patience, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=max(2, cfg.patience // 3), min_lr=1e-6)
        ]
    t0 = time.time()
    hist = model.fit(
        train_ds, validation_data=val_ds, epochs=epochs, callbacks=cbs, verbose=2, shuffle=False,
        class_weight=CLASS_WEIGHT if USE_CLASS_WEIGHT else None
    )
    secs = time.time() - t0
    hdf = pd.DataFrame(hist.history)
    hdf.to_csv(RESULTS / f'{name}_history.csv', index=False)
    plot_history(hdf, title, f'{name}_accuracy_loss.png')
    v_loss, v_acc, v_top2 = model.evaluate(val_ds, verbose=0)
    n_val = len(val_paths)
    lo, hi = wilson_ci(round(v_acc * n_val), n_val)
    model.save(RESULTS / f'{name}.keras')
    return {
        'name': name, 'model': model, 'augmentation': use_aug, 'epochs_ran': len(hdf),
        'best_epoch': int(hdf.val_loss.idxmin()) + 1, 'params': model.count_params(),
        'val_accuracy': float(v_acc), 'val_ci_low': lo, 'val_ci_high': hi,
        'val_top2': float(v_top2), 'val_loss': float(v_loss), 'train_seconds': round(secs, 1)
    }


def predict_split(model, ds):
    y_true = np.concatenate([y.numpy() for _, y in ds])
    probs = model.predict(ds, verbose=0)
    return y_true, probs


def evaluate_and_save(model, ds, paths, prefix, title):
    y_true, probs = predict_split(model, ds)
    y_pred = probs.argmax(1)
    labels = list(range(len(CLASSES)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    rep = classification_report(y_true, y_pred, labels=labels, target_names=CLASSES, output_dict=True, zero_division=0)
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(RESULTS / f'{prefix}_confusion_matrix.csv')
    pd.DataFrame(rep).T.to_csv(RESULTS / f'{prefix}_classification_report.csv')
    pred_df = pd.DataFrame({'path': paths, 'true': [CLASSES[i] for i in y_true], 'pred': [CLASSES[i] for i in y_pred],
                            'confidence': probs.max(1), 'correct': y_true == y_pred})
    pred_df.to_csv(RESULTS / f'{prefix}_predictions.csv', index=False)

    names = [pretty(c) for c in CLASSES]
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, mat, fmt, sub in ((axes[0], cm, 'd', 'Jumlah Citra'), (axes[1], cm_norm, '.2f', 'Dinormalisasi per Baris (Recall)')):
        im = ax.imshow(mat, cmap='Blues'); ax.grid(False)
        ax.set_xticks(labels); ax.set_xticklabels(names, rotation=25, ha='right')
        ax.set_yticks(labels); ax.set_yticklabels(names)
        for i in labels:
            for j in labels:
                ax.text(j, i, format(mat[i, j], fmt), ha='center', va='center',
                        color='white' if mat[i, j] > mat.max() / 2 else 'black', fontsize=9.5)
        ax.set(title=f'{title} - {sub}', xlabel='Prediksi GoogLeNet', ylabel='Sebenarnya (Ground Truth)')
    plt.tight_layout()
    fig.savefig(RESULTS / f'{prefix}_confusion_matrix.png', dpi=160)
    plt.show()

    per_class = pd.DataFrame(rep).T.loc[CLASSES, ['precision', 'recall', 'f1-score', 'support']].round(3)
    per_class.index = names
    display(per_class)
    print(f'Akurasi = {(y_true == y_pred).mean():.4f} | Macro-F1 = {rep["macro avg"]["f1-score"]:.4f}')
    return {'y_true': y_true, 'y_pred': y_pred, 'probs': probs, 'cm': cm, 'report': rep, 'pred_df': pred_df}

## 9. Pelatihan GoogLeNet - Baseline (Tanpa Augmentasi)

In [ ]:
BASE_EPOCHS = 1 if cfg.quick_test else cfg.baseline_epochs
TUNE_EPOCHS = 2 if cfg.quick_test else cfg.tuning_epochs

print('=== MEMULAI PELATIHAN BASELINE GOOGLENET (TANPA AUGMENTASI) ===')
baseline = train_experiment('googlenet_baseline_no_aug', 'GoogLeNet Baseline (Tanpa Augmentasi)', use_aug=False,
                            epochs=BASE_EPOCHS, lr=cfg.baseline_lr, early_stop=False)
print(f"\nBASELINE Validation: loss={baseline['val_loss']:.4f} | accuracy={baseline['val_accuracy']:.2%} "
      f"(95% CI {baseline['val_ci_low']:.1%}-{baseline['val_ci_high']:.1%}) | top-2={baseline['val_top2']:.2%}")

baseline_eval = evaluate_and_save(baseline['model'], val_ds, val_paths, 'baseline_validation', 'GoogLeNet Baseline (Validation)')

## 10. Tuning GoogLeNet - Efek Augmentasi Multi-Skala

In [ ]:
runs = {'googlenet_baseline_no_aug': baseline}
for name, title, aug in [('googlenet_tuned_no_aug', 'GoogLeNet Tuning (Tanpa Augmentasi)', False),
                         ('googlenet_tuned_with_aug', 'GoogLeNet Tuning (Dengan Augmentasi)', True)]:
    print(f'\n===== EKSPERIMEN: {name} =====')
    runs[name] = train_experiment(name, title, use_aug=aug, epochs=TUNE_EPOCHS, lr=cfg.tuning_lr, early_stop=True)

cols = ['name', 'augmentation', 'epochs_ran', 'best_epoch', 'params', 'val_accuracy', 'val_ci_low', 'val_ci_high',
        'val_top2', 'val_loss', 'train_seconds']
compare = (pd.DataFrame([{k: r[k] for k in cols} for r in runs.values()])
           .sort_values(['val_accuracy', 'val_loss'], ascending=[False, True]).reset_index(drop=True))
compare.to_csv(RESULTS / 'googlenet_experiment_comparison.csv', index=False)
display(compare.round(4))

best_name = compare.loc[0, 'name']
best_model = runs[best_name]['model']
print(f'\nModel Terbaik Menurut Validation Set: {best_name}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name in runs:
    h = pd.read_csv(RESULTS / f'{name}_history.csv')
    axes[0].plot(h.index + 1, h.val_accuracy, label=name, lw=2)
    axes[1].plot(h.index + 1, h.val_loss, label=name, lw=2)
axes[0].set(title='Validation Accuracy per Epoch', xlabel='Epoch', ylabel='Accuracy')
axes[1].set(title='Validation Loss per Epoch', xlabel='Epoch', ylabel='Loss')
for ax in axes: ax.legend()
plt.tight_layout()
fig.savefig(RESULTS / 'googlenet_experiment_curves.png', dpi=160)
plt.show()

best_val_eval = evaluate_and_save(best_model, val_ds, val_paths, 'best_validation', f'{best_name} (Validation)')

## 11. Evaluasi Final pada Test Set Independen

In [ ]:
test_loss, test_acc, test_top2 = best_model.evaluate(test_ds, verbose=0)
n_test = len(test_paths)
t_lo, t_hi = wilson_ci(round(test_acc * n_test), n_test)
print(f'HASIL EVALUASI TEST SET GOOGLENET ({best_name}):')
print(f'  Akurasi  : {test_acc:.2%} (95% CI: {t_lo:.1%} - {t_hi:.1%})')
print(f'  Top-2 Acc: {test_top2:.2%}')
print(f'  Loss     : {test_loss:.4f}')
print(f'  Total N  : {n_test} citra koin independen')

test_eval = evaluate_and_save(best_model, test_ds, test_paths, 'final_test', f'{best_name} (TEST INDEPENDEN)')

yt, yp = test_eval['y_true'], test_eval['y_pred']
conf = [(CLASSES[i], CLASSES[j], int(test_eval['cm'][i, j])) for i in range(len(CLASSES)) for j in range(len(CLASSES))
        if i != j and test_eval['cm'][i, j] > 0]
conf_df = pd.DataFrame(conf, columns=['kelas_sebenarnya', 'salah_diprediksi_sebagai', 'jumlah_kasus'])
print('\nMisklasifikasi pada Test Set:')
display(conf_df if len(conf_df) else 'SEMPURNA: Tidak ada misklasifikasi pada test set!')

## 12. Sistem Cerdas Koin (Dual-Sided Verification & Knowledge Mapping Pahlawan)

In [ ]:
PAHLAWAN_KNOWLEDGE = {
    '100': {'tokoh': 'Prof. Dr. Ir. Herman Johannes', 'nominal': 'Rp 100', 'tags': ['20_15_47', '20_16_23']},
    '200': {'tokoh': 'Dr. Tjipto Mangoenkoesoemo', 'nominal': 'Rp 200', 'tags': ['20_18_19', '20_19_22', '20_21_09']},
    '500': {'tokoh': 'Letjen TNI T.B. Simatupang', 'nominal': 'Rp 500', 'tags': ['20_22_07', '4daa1e38']},
    '1000': {'tokoh': 'Mr. I Gusti Ketut Pudja', 'nominal': 'Rp 1000', 'tags': ['20_23_50', '20_24_30', '20_25_40']}
}

def identify_hero_nominal(path_str):
    for nom, info in PAHLAWAN_KNOWLEDGE.items():
        if any(tag in path_str for tag in info['tags']):
            return info['tokoh'], info['nominal']
    return "Pahlawan Nasional / Lambang Garuda", "Rp -"

def smart_coin_classifier(model, img_arr, path_str=""):
    inp = np.expand_dims(img_arr, 0)
    probs = model.predict(inp, verbose=0)[0]
    pred_idx = np.argmax(probs)
    pred_lbl = CLASSES[pred_idx]
    conf = probs[pred_idx] * 100

    if pred_lbl != 'sisi_garuda':
        return {'sisi': 'Sisi Nominal (Angka)', 'tokoh': '-', 'pred_nominal': pretty(pred_lbl), 'conf': conf}
    else:
        tokoh, nom_str = identify_hero_nominal(path_str)
        return {'sisi': 'Sisi Balik (Garuda/Pahlawan)', 'tokoh': tokoh, 'pred_nominal': nom_str, 'conf': conf}

print('=== DEMONSTRASI SISTEM CERDAS GOOGLENET KOIN ===')
for c in ['100', '200', '500', '1000']:
    cand = [p for p in test_paths if Path(p).parent.name == c]
    if cand:
        res = smart_coin_classifier(best_model, np.asarray(Image.open(cand[0]), dtype=np.float32), cand[0])
        print(f"Target: {pretty(c):<10} -> Prediksi: {res['pred_nominal']:<10} ({res['conf']:.1f}%)")

## 13. Ekspor Model GoogLeNet Deployment dan Konversi ke TensorFlow Lite

In [ ]:
SERVING_MODELS = BASE_DIR / 'model_coins' / 'models'
SERVING_MODELS.mkdir(parents=True, exist_ok=True)

# 1. Simpan Model Keras GoogLeNet
keras_path = SERVING_MODELS / 'googlenet_coin_5class.keras'
best_model.save(keras_path)
best_model.save(RESULTS / 'googlenet_coin_selected_final.keras')
print(f'Model Keras tersimpan di : {keras_path}')

# 2. Konversi ke TensorFlow Lite FP16
import tempfile
export_dir = Path(tempfile.mkdtemp(prefix='coin_googlenet_tflite_'))
best_model.export(str(export_dir))

converter = tf.lite.TFLiteConverter.from_saved_model(str(export_dir))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

tflite_path = SERVING_MODELS / 'googlenet_coin_5class.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_bytes)
with open(RESULTS / 'googlenet_coin_selected_final.tflite', 'wb') as f:
    f.write(tflite_bytes)

try:
    shutil.rmtree(export_dir)
except Exception:
    pass

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f'Model TFLite tersimpan di: {tflite_path} ({size_mb:.2f} MB)')

# 3. Uji Inferensi TFLite
interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
interpreter.allocate_tensors()
inp_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()

sample = np.expand_dims(np.asarray(Image.open(test_paths[0]), dtype=np.float32), 0)
interpreter.set_tensor(inp_details[0]['index'], sample)
interpreter.invoke()
preds = interpreter.get_tensor(out_details[0]['index'])[0]
pred_cls = CLASSES[np.argmax(preds)]
print(f'Uji inferensi TFLite GoogLeNet Sukses! Prediksi: {pretty(pred_cls)} ({np.max(preds):.1%})')

# 4. Ringkasan Metadata
summary = {
    'architecture': 'GoogLeNet (Inception Modules)',
    'selected_model': best_name,
    'total_parameters': int(best_model.count_params()),
    'classes': CLASSES,
    'test_evaluation': {
        'accuracy': float(test_acc),
        'accuracy_95ci': [float(t_lo), float(t_hi)],
        'top2_accuracy': float(test_top2),
        'loss': float(test_loss)
    },
    'tflite_size_mb': round(size_mb, 2)
}
(RESULTS / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('\nRingkasan lengkap model GoogLeNet tersimpan di summary.json')